<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-01-classical-ml-statsrefresher/k-means-clustering/notebooks/exercises-0_5_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ Lesson 0.5.2: K-Means Clustering — Practice Exercises

**Netsetos GenAI Course** | Module 0.5 — Classical ML & Stats Refresher

6 exercises covering K-Means from scratch, elbow/silhouette, K-Means++ vs random, DBSCAN comparison, FAISS IVF simulation, and a full RAG index pipeline.

**Key insight:** K-Means IS the algorithm behind production vector search (FAISS IVF). Understanding it = understanding how RAG works.

---

## Exercise 1 (Easy): K-Means from Scratch

### 📚 Theory
4 steps: (1) init centroids, (2) assign points to nearest, (3) update centroids to means, (4) repeat until convergence. Same algorithm FAISS uses at scale.

### 📋 Task
1. Generate 300 points in 3 clusters
2. Implement K-Means with NumPy only
3. Track inertia per iteration
4. Verify against sklearn

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 1
import numpy as np
from sklearn.datasets import make_blobs

np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

def kmeans_scratch(X, k, max_iters=100):
    idx = np.random.choice(len(X), k, replace=False)
    centroids = X[idx].copy()
    # TODO: implement assign → update → convergence loop
    pass

# TODO: run and compare with sklearn

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

def kmeans_scratch(X, k, max_iters=100):
    """K-Means from scratch using only NumPy."""
    idx = np.random.choice(len(X), k, replace=False)
    centroids = X[idx].copy()

    for iteration in range(max_iters):
        # Step 2: Assign to nearest centroid
        distances = np.linalg.norm(X[:, None] - centroids, axis=2)
        labels = distances.argmin(axis=1)

        # Step 3: Update centroids
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])

        # Inertia
        inertia = sum(np.sum((X[labels == i] - new_centroids[i])**2) for i in range(k))

        # Step 4: Convergence check
        if np.allclose(centroids, new_centroids):
            print(f'Converged at iteration {iteration}')
            break
        centroids = new_centroids

    return labels, centroids, inertia

labels, centroids, inertia = kmeans_scratch(X, k=3)
print(f'Cluster sizes: {np.bincount(labels)}')
print(f'Inertia (scratch): {inertia:.2f}')

# Verify with sklearn
km = KMeans(3, random_state=42, n_init=10).fit(X)
print(f'Inertia (sklearn): {km.inertia_:.2f}')

</details>

---

## Exercise 2 (Easy): Elbow + Silhouette Method

### 📚 Theory
Elbow: plot inertia vs K, pick the bend. Silhouette: -1 to +1 (higher = better separated). Use BOTH to find optimal K. In production, this tunes FAISS's nlist.

### 📋 Task
1. Generate 4-cluster data
2. K=2 through 8 — inertia + silhouette
3. Identify optimal K

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 2
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=500, centers=4, random_state=42)

# TODO: K=2..8, inertia + silhouette, find optimal K

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=500, centers=4, random_state=42)

print(f'{"K":>3} {"Inertia":>10} {"Silhouette":>11}')
print('-' * 30)
best_k, best_sil = 2, -1
for k in range(2, 9):
    km = KMeans(k, random_state=42, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    marker = ' ← optimal' if sil > best_sil else ''
    if sil > best_sil:
        best_sil = sil
        best_k = k
    print(f'  {k:>2} {km.inertia_:>10.0f} {sil:>10.3f}{marker}')

print(f'\nOptimal K={best_k} (silhouette={best_sil:.3f})')

</details>

---

## Exercise 3 (Medium): K-Means++ vs Random Initialization

### 📚 Theory
Random init can place centroids in the same cluster → bad local minimum. K-Means++ spreads centroids proportional to distance. Run multiple trials to prove the difference statistically.

### 📋 Task
1. Generate 5-cluster data
2. 10 runs each: random init vs k-means++
3. Compare avg inertia, worst inertia, avg iterations, consistency

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 3
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import numpy as np

X, _ = make_blobs(n_samples=1000, centers=5, random_state=42)

# TODO: 10 runs each, random vs k-means++, compare

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import numpy as np

X, _ = make_blobs(n_samples=1000, centers=5, random_state=42)

results = {}
for init_method in ['random', 'k-means++']:
    inertias, iters = [], []
    for seed in range(10):
        km = KMeans(5, init=init_method, n_init=1, random_state=seed, max_iter=300).fit(X)
        inertias.append(km.inertia_)
        iters.append(km.n_iter_)
    results[init_method] = {
        'avg_inertia': np.mean(inertias),
        'worst_inertia': np.max(inertias),
        'std_inertia': np.std(inertias),
        'avg_iters': np.mean(iters),
    }

print(f'{"Metric":<20} {"Random":>12} {"K-Means++":>12}')
print('-' * 48)
for metric in ['avg_inertia', 'worst_inertia', 'std_inertia', 'avg_iters']:
    r = results['random'][metric]
    k = results['k-means++'][metric]
    winner = ' ← better' if k < r else ''
    print(f'  {metric:<18} {r:>11.1f} {k:>11.1f}{winner}')

print(f'\nK-Means++ wins: lower avg inertia, much lower variance, fewer iterations.')

</details>

---

## Exercise 4 (Medium): K-Means vs DBSCAN

### 📚 Theory
K-Means forces spherical clusters. DBSCAN groups by density — handles moons, circles, arbitrary shapes, AND detects outliers as noise.

### 📋 Task
1. Generate moons and circles datasets
2. Run K-Means and DBSCAN on both
3. Compare silhouette scores
4. Show where K-Means fails

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 4
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_moons, make_circles
from sklearn.metrics import silhouette_score

X_m, _ = make_moons(500, noise=0.1, random_state=42)
X_c, _ = make_circles(500, noise=0.05, factor=0.5, random_state=42)

# TODO: KMeans + DBSCAN on both, compare silhouette

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_moons, make_circles
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import numpy as np

datasets = {
    'Moons': make_moons(500, noise=0.1, random_state=42)[0],
    'Circles': make_circles(500, noise=0.05, factor=0.5, random_state=42)[0],
}

print(f'{"Dataset":<10} {"K-Means sil":>12} {"DBSCAN sil":>12} {"Winner":<10}')
print('-' * 50)
for name, X in datasets.items():
    X_s = StandardScaler().fit_transform(X)

    # K-Means
    km = KMeans(2, random_state=42, n_init=10).fit(X_s)
    sil_km = silhouette_score(X_s, km.labels_)

    # DBSCAN
    db = DBSCAN(eps=0.3, min_samples=5).fit(X_s)
    mask = db.labels_ != -1
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    sil_db = silhouette_score(X_s[mask], db.labels_[mask]) if mask.sum() > 1 and n_clusters > 1 else 0
    noise = (db.labels_ == -1).sum()

    winner = 'DBSCAN' if sil_db > sil_km else 'K-Means'
    print(f'  {name:<8} {sil_km:>11.3f} {sil_db:>11.3f}  {winner} ({noise} noise pts)')

print(f'\nDBSCAN handles non-spherical shapes that K-Means cannot.')

</details>

---

## Exercise 5 (Hard): FAISS IVF Simulator

### 📚 Theory
FAISS IVF: K-Means at index time → inverted lists. Query time: find nearest nprobe centroids → search only those lists. Speedup = nlist/nprobe. This powers every production RAG system.

### 📋 Task
1. 10K embeddings (128-dim), normalized
2. K-Means with nlist=64 → build inverted lists
3. Brute-force baseline
4. IVF search for nprobe = 1, 4, 8, 16, 32
5. Measure speedup and recall@10

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 5
import numpy as np
from sklearn.cluster import KMeans
import time

np.random.seed(42)
n_docs, dim = 10000, 128
embs = np.random.randn(n_docs, dim).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)

query = np.random.randn(dim).astype(np.float32)
query /= np.linalg.norm(query)

nlist = 64

# TODO: K-Means index → inverted lists
# TODO: brute-force baseline
# TODO: IVF search at different nprobe values
# TODO: speedup and recall@10

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import time

np.random.seed(42)
n_docs, dim = 10000, 128
embs = np.random.randn(n_docs, dim).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)

query = np.random.randn(dim).astype(np.float32)
query /= np.linalg.norm(query)

nlist = 64

# Index: K-Means
print('Building index...')
t0 = time.time()
km = KMeans(nlist, random_state=42, n_init=1).fit(embs)
print(f'Index built in {time.time()-t0:.2f}s\n')

# Inverted lists
inv_lists = {i: np.where(km.labels_ == i)[0] for i in range(nlist)}

# Brute-force baseline
t0 = time.time()
all_scores = embs @ query
bf_top10 = set(all_scores.argsort()[-10:][::-1])
bf_time = (time.time() - t0) * 1000
print(f'Brute-force: {n_docs} comparisons, {bf_time:.2f}ms\n')

# IVF search
centroid_scores = km.cluster_centers_ @ query

print(f'{"nprobe":>6} {"Searched":>8} {"Speedup":>8} {"Recall@10":>10} {"Time(ms)":>9}')
print('-' * 48)
for nprobe in [1, 4, 8, 16, 32]:
    top_clusters = centroid_scores.argsort()[-nprobe:][::-1]

    t0 = time.time()
    candidates = np.concatenate([inv_lists[c] for c in top_clusters])
    cand_scores = embs[candidates] @ query
    ivf_top10_idx = candidates[cand_scores.argsort()[-10:][::-1]]
    ivf_time = (time.time() - t0) * 1000

    ivf_top10 = set(ivf_top10_idx)
    recall = len(bf_top10 & ivf_top10) / 10
    speedup = n_docs / len(candidates)

    print(f'  {nprobe:>4} {len(candidates):>8} {speedup:>7.1f}x {recall:>9.0%} {ivf_time:>8.2f}')

print(f'\nSweet spot: nprobe=8-16 for >90% recall with 4-8x speedup')

</details>

---

## Exercise 6 (Hard): Full RAG Index Pipeline

### 📚 Theory
Combine everything: embeddings → K-Means index → inverted lists → query → recall + latency measurement. This is the exact pipeline in Pinecone, Qdrant, FAISS.

### 📋 Task
1. 50K embeddings (256-dim)
2. Test nlist=32,64,128,256 × nprobe=1,4,8,16
3. Measure build time, query time, speedup, recall@10
4. Recommend best config for >95% recall

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 6
import numpy as np
from sklearn.cluster import KMeans
import time

np.random.seed(42)
n_docs, dim = 50000, 256
embs = np.random.randn(n_docs, dim).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)
query = np.random.randn(dim).astype(np.float32)
query /= np.linalg.norm(query)

# TODO: brute-force baseline
# TODO: test multiple nlist × nprobe combinations
# TODO: full diagnostic report + recommendation

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import time

np.random.seed(42)
n_docs, dim = 50000, 256
embs = np.random.randn(n_docs, dim).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)
query = np.random.randn(dim).astype(np.float32)
query /= np.linalg.norm(query)

# Brute-force
t0 = time.time()
all_scores = embs @ query
bf_top10 = set(all_scores.argsort()[-10:][::-1])
bf_ms = (time.time() - t0) * 1000

print('═' * 65)
print('  RAG INDEX PIPELINE REPORT')
print('═' * 65)
print(f'\nBrute-force: {n_docs} docs, {bf_ms:.1f}ms\n')
print(f'{"nlist":>6} {"nprobe":>7} {"Build(s)":>9} {"Query(ms)":>10} {"Speedup":>8} {"Recall":>7}')
print('-' * 55)

best_config = None
best_speedup = 0

for nlist in [64, 128]:
    t0 = time.time()
    km = KMeans(nlist, random_state=42, n_init=1, max_iter=50).fit(embs)
    build_s = time.time() - t0
    inv = {i: np.where(km.labels_ == i)[0] for i in range(nlist)}
    c_scores = km.cluster_centers_ @ query

    for nprobe in [4, 8, 16]:
        top_c = c_scores.argsort()[-nprobe:][::-1]
        t0 = time.time()
        cands = np.concatenate([inv[c] for c in top_c])
        scores = embs[cands] @ query
        ivf_top10 = set(cands[scores.argsort()[-10:][::-1]])
        q_ms = (time.time() - t0) * 1000

        recall = len(bf_top10 & ivf_top10) / 10
        speedup = bf_ms / max(q_ms, 0.001)

        marker = ''
        if recall >= 0.9 and (best_config is None or speedup > best_speedup):
            best_config = (nlist, nprobe)
            best_speedup = speedup
            marker = ' ★'

        print(f'  {nlist:>4} {nprobe:>7} {build_s:>8.1f} {q_ms:>9.2f} {speedup:>7.1f}x {recall:>6.0%}{marker}')

print(f'\n  RECOMMENDATION: nlist={best_config[0]}, nprobe={best_config[1]}')
print(f'  → {best_speedup:.1f}x faster than brute-force with ≥90% recall')

</details>

---

## 🎉 Done!

You've built the vector search infrastructure behind every production RAG system:
- **K-Means from scratch** — the 4-step algorithm
- **Elbow + Silhouette** — choosing optimal K
- **K-Means++** — why smart init matters
- **DBSCAN** — when K-Means fails
- **FAISS IVF** — K-Means powering vector search
- **Full pipeline** — 50K docs, speedup + recall measurement

This is exactly what you'll use in **Module 6 (RAG)** and **Module 10 (scaling).** Next: **Lesson 0.5.3.**